# 🔍 Módulo 2. Validación de datos

## Objetivo

Este notebook evalúa la calidad de los datos originales antes de realizar cualquier transformación.

Las validaciones permiten identificar posibles problemas de calidad, inconsistencias y características de cada dataset que deberán tenerse en cuenta durante la etapa de limpieza.

## Flujo del notebook

El proceso de validación se desarrolla mediante las siguientes etapas:

1. Cargar los datasets originales.
2. Inicializar el validador.
3. Analizar dimensiones y tipos de datos.
4. Identificar valores faltantes.
5. Buscar registros duplicados.
6. Evaluar posibles llaves para integración.
7. Detectar columnas numéricas almacenadas como texto.
8. Documentar las decisiones para la etapa de limpieza.

In [35]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent

sys.path.append(str(PROJECT_ROOT))

from src.data.loader import DataLoader
from src.preprocessing.validator import DataValidator

## Inicialización del validador

Se crea una instancia de `DataValidator`, responsable de inspeccionar los datasets sin modificar su contenido.

In [36]:
DATA_DIR = PROJECT_ROOT / "data" / "raw"

loader = DataLoader(DATA_DIR)
validator = DataValidator()

📂 Carpeta de datos: /Users/alejandraarciniegas/Documents/educational_crisis_ai/data/raw


In [37]:
datasets = loader.load_all()

principal = datasets["principal"]
pae = datasets["pae"]
etc = datasets["etc"]


📄 Cargando principal.csv...
✅ Carga completada
   Filas: 15,707
   Columnas: 41

📄 Cargando pae.csv...
✅ Carga completada
   Filas: 121,379
   Columnas: 10

📄 Cargando etc.csv...
✅ Carga completada
   Filas: 1,336
   Columnas: 37


## Dataset principal

### Tipos de datos

Se inspeccionan los tipos de cada columna para identificar variables que requieran conversión durante la limpieza.

In [4]:
validator.column_summary(principal)

,dtype,missing,missing_%,unique_values
AÑO,int64,0,0.00,14
CÓDIGO_MUNICIPIO,int64,0,0.00,1124
MUNICIPIO,object,0,0.00,1038
CÓDIGO_DEPARTAMENTO,int64,0,0.00,34
DEPARTAMENTO,object,0,0.00,36
CÓDIGO_ETC,float64,0,0.00,194
ETC,object,0,0.00,100
POBLACIÓN_5_16,object,6,0.04,8884
TASA_MATRICULACIÓN_5_16,object,115,0.73,5994
COBERTURA_NETA,object,111,0.71,5489


### Valores faltantes

Se identifican las columnas que presentan datos ausentes y la magnitud del problema en cada una.

In [5]:
validator.missing_values(principal)

TAMAÑO_PROMEDIO_DE_GRUPO       8135
SEDES_CONECTADAS_A_INTERNET    7939
DESERCIÓN_TRANSICIÓN            903
DESERCIÓN_MEDIA                 734
DESERCIÓN_SECUNDARIA            270
DESERCIÓN_PRIMARIA              242
REPITENCIA_TRANSICIÓN           159
REPITENCIA_SECUNDARIA           152
REPITENCIA_PRIMARIA             148
REPROBACIÓN_MEDIA               145
REPITENCIA                      143
DESERCIÓN                       142
REPITENCIA_MEDIA                139
COBERTURA_BRUTA_MEDIA           127
TASA_MATRICULACIÓN_5_16         115
COBERTURA_NETA                  111
REPROBACIÓN_SECUNDARIA          106
APROBACIÓN_MEDIA                101
REPROBACIÓN_PRIMARIA             97
COBERTURA_BRUTA_TRANSICIÓN       97
COBERTURA_NETA_SECUNDARIA        94
APROBACIÓN_TRANSICIÓN            93
REPROBACIÓN_TRANSICIÓN           93
COBERTURA_NETA_MEDIA             93
COBERTURA_NETA_PRIMARIA          91
COBERTURA_BRUTA_SECUNDARIA       88
REPROBACIÓN                      86
COBERTURA_BRUTA_PRIMARIA    

### Llaves candidatas

Se evalúan posibles combinaciones de columnas que puedan utilizarse posteriormente durante la integración de los datasets.

In [6]:
validator.validate_keys(
    principal,
    ["AÑO", "CÓDIGO_MUNICIPIO"]
)

{'keys': ['AÑO', 'CÓDIGO_MUNICIPIO'],
 'rows': 15707,
 'unique_combinations': 15707,
 'duplicated_keys': np.int64(0),
 'has_nulls': False,
 'null_values': AÑO                 0
 CÓDIGO_MUNICIPIO    0
 Name: null_values, dtype: int64}

### Candidatos a conversión numérica

Se detectan columnas almacenadas como texto cuyo contenido corresponde principalmente a valores numéricos.

In [7]:
validator.numeric_candidates(principal)

,column,current_dtype,conversion_rate_%,contains_percent,sample
0,POBLACIÓN_5_16,object,100.0,False,499
1,TASA_MATRICULACIÓN_5_16,object,100.0,True,56.11%
2,COBERTURA_NETA,object,100.0,True,56.11%
3,COBERTURA_NETA_TRANSICIÓN,object,100.0,True,39.53%
4,COBERTURA_NETA_PRIMARIA,object,100.0,True,59.13%
5,COBERTURA_NETA_SECUNDARIA,object,100.0,True,51.52%
6,COBERTURA_NETA_MEDIA,object,100.0,True,26.51%
7,COBERTURA_BRUTA,object,100.0,True,61.92%
8,COBERTURA_BRUTA_TRANSICIÓN,object,100.0,True,58.14%
9,COBERTURA_BRUTA_PRIMARIA,object,100.0,True,67.79%


In [8]:
validator.sample_values(
    principal,
    "DESERCIÓN"
)

array(['0%', '3.24%', '5.5%', '6.3%', '5.16%', '8.45%', '8.39%', '7.83%',
       '3.54%', '5.06%'], dtype=object)

In [28]:
validator.sample_values(
    principal,
    "TAMAÑO_PROMEDIO_DE_GRUPO"
)

array(['17%', '25.632%', '21.571%', '28.628%', '10.545%', '9.326%',
       '10.767%', '16.786%', '15.757%', '22.625%'], dtype=object)

In [9]:
validator.sample_values(
    principal,
    "COBERTURA_NETA"
)

array(['56.11%', '95.33%', '50.7%', '81.42%', '90.96%', '148.17%',
       '46.26%', '0%', '19.95%', '54.8%'], dtype=object)

In [10]:
validator.sample_values(
    principal,
    "POBLACIÓN_5_16"
)

array(['499', '1862', '25239', '1157', '2645', '4600', '562', '208',
       '802', '1666'], dtype=object)

## Dataset PAE

In [11]:
validator.column_summary(pae)

,dtype,missing,missing_%,unique_values
FECHA,object,0,0.0,5
FECHA_CORTE,object,0,0.0,5
CODIGO_DEPARTAMENTO,int64,0,0.0,33
DEPARTAMENTO,object,0,0.0,35
CODIGO_MUNICIPIO,int64,0,0.0,1121
MUNICIPIO,object,0,0.0,1281
ZONA,object,0,0.0,4
JORNADA,object,0,0.0,2
GRUPO_POBLACIONAL,object,0,0.0,10
CANTIDAD_BENEFICIARIOS_PAE,object,0,0.0,3514


In [12]:
validator.missing_values(pae)

Series([], dtype: int64)

In [13]:
validator.duplicated_rows(pae)

np.int64(42412)

In [14]:
validator.validate_keys(
    pae,
    ["CODIGO_MUNICIPIO", "FECHA"]
)

{'keys': ['CODIGO_MUNICIPIO', 'FECHA'],
 'rows': 121379,
 'unique_combinations': 5521,
 'duplicated_keys': np.int64(115858),
 'has_nulls': False,
 'null_values': CODIGO_MUNICIPIO    0
 FECHA               0
 Name: null_values, dtype: int64}

In [22]:
validator.numeric_candidates(pae)

,column,current_dtype,conversion_rate_%,contains_percent,sample
0,CANTIDAD_BENEFICIARIOS_PAE,object,100.0,False,"187,953"


In [15]:
validator.sample_values(
    pae,
    "CANTIDAD_BENEFICIARIOS_PAE"
)

array(['187,953', '311', '1,348', '2,365', '34,815', '67', '267', '315',
       '5', '3'], dtype=object)

In [16]:
validator.sample_values(
    pae,
    "FECHA"
)

array(['31/12/2020', '30/04/2022', '30/06/2021', '31/10/2021',
       '31/12/2021'], dtype=object)

## Dataset ETC

In [17]:
validator.column_summary(etc)

,dtype,missing,missing_%,unique_values
AÑO,int64,0,0.00,14
CÓDIGO_ETC,int64,0,0.00,97
ETC,object,0,0.00,97
POBLACIÓN_5_16,object,0,0.00,1327
TASA_MATRICULACIÓN_5_16,object,1,0.07,1156
COBERTURA_NETA,float64,0,0.00,1134
COBERTURA_NETA_TRANSICIÓN,float64,0,0.00,1166
COBERTURA_NETA_PRIMARIA,float64,0,0.00,1144
COBERTURA_NETA_SECUNDARIA,float64,0,0.00,1150
COBERTURA_NETA_MEDIA,float64,0,0.00,1141


In [18]:
validator.missing_values(etc)

SEDES_CONECTADAS_A_INTERNET    676
TAMAÑO_PROMEDIO_DE_GRUPO       675
REPROBACIÓN_TRANSICIÓN          32
REPROBACIÓN_SECUNDARIA           8
REPROBACIÓN_PRIMARIA             7
DESERCIÓN_TRANSICIÓN             4
REPROBACIÓN                      3
REPROBACIÓN_MEDIA                3
REPITENCIA_TRANSICIÓN            2
TASA_MATRICULACIÓN_5_16          1
REPITENCIA_SECUNDARIA            1
REPITENCIA_PRIMARIA              1
REPITENCIA                       1
APROBACIÓN_SECUNDARIA            1
APROBACIÓN_MEDIA                 1
APROBACIÓN_PRIMARIA              1
APROBACIÓN_TRANSICIÓN            1
APROBACIÓN                       1
DESERCIÓN_MEDIA                  1
DESERCIÓN_SECUNDARIA             1
DESERCIÓN_PRIMARIA               1
DESERCIÓN                        1
REPITENCIA_MEDIA                 1
dtype: int64

In [19]:
validator.duplicated_rows(etc)

np.int64(0)

In [20]:
validator.validate_keys(
    etc,
    ["AÑO", "CÓDIGO_ETC"]
)

{'keys': ['AÑO', 'CÓDIGO_ETC'],
 'rows': 1336,
 'unique_combinations': 1335,
 'duplicated_keys': np.int64(1),
 'has_nulls': False,
 'null_values': AÑO           0
 CÓDIGO_ETC    0
 Name: null_values, dtype: int64}

In [23]:
validator.numeric_candidates(etc)

,column,current_dtype,conversion_rate_%,contains_percent,sample
0,POBLACIÓN_5_16,object,100.0,False,27144
1,TASA_MATRICULACIÓN_5_16,object,100.0,False,100.58
2,TAMAÑO_PROMEDIO_DE_GRUPO,object,100.0,False,"32,377"
3,SEDES_CONECTADAS_A_INTERNET,object,100.0,True,7.59%


In [21]:
validator.sample_values(
    etc,
    "DESERCIÓN"
)

array([3.93, 4.58, 4.47, 2.65, 2.69, 2.49, 3.4 , 3.94, 2.37, 2.82])

In [26]:
etc[etc.duplicated(subset=["AÑO", "CÓDIGO_ETC"], keep=False)]

,AÑO,CÓDIGO_ETC,ETC,POBLACIÓN_5_16,TASA_MATRICULACIÓN_5_16,COBERTURA_NETA,COBERTURA_NETA_TRANSICIÓN,COBERTURA_NETA_PRIMARIA,COBERTURA_NETA_SECUNDARIA,COBERTURA_NETA_MEDIA,...,REPROBACIÓN,REPROBACIÓN_TRANSICIÓN,REPROBACIÓN_PRIMARIA,REPROBACIÓN_SECUNDARIA,REPROBACIÓN_MEDIA,REPITENCIA,REPITENCIA_TRANSICIÓN,REPITENCIA_PRIMARIA,REPITENCIA_SECUNDARIA,REPITENCIA_MEDIA
423,2020,3816,Ibagué,89880,106.82,106.60,83.05,106.65,93.36,55.41,...,5.95,0.19,3.28,9.95,5.56,4.93,0.39,3.29,8.21,2.99
481,2020,3816,Ibagué,89880,1.07,1.07,0.83,1.07,0.93,0.55,...,0.06,0.00,0.03,0.10,0.06,0.05,0.00,0.03,0.08,0.03


# Conclusiones de la validación

A partir del análisis realizado se identificaron las siguientes observaciones principales:

- Existen columnas numéricas almacenadas como texto debido a la presencia de porcentajes y distintos formatos numéricos.
- Algunos indicadores presentan valores faltantes que deberán conservarse o tratarse dependiendo de su naturaleza.
- El dataset PAE contiene múltiples registros por municipio, diferenciados por variables como zona, jornada y grupo poblacional, por lo que requerirá una agregación antes de integrarse con el resto de la información.
- El dataset ETC presenta un registro duplicado conocido para el año 2020, el cual será eliminado durante la etapa de limpieza.
- No se encontraron problemas que impidan continuar con el proceso de depuración.

Estas observaciones servirán como base para implementar las transformaciones del Notebook 3.